In [ ]:
import pandas as pd
from palmerpenguins import load_penguins

# Cargamos los datos
df = load_penguins()
df.head()

In [ ]:
# Dropeamos filas incompletas de las variables numericas
df.dropna(subset=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"], inplace=True)

In [ ]:
df.groupby("species").get_group("Adelie").describe()

In [ ]:
df.groupby("species").get_group("Chinstrap").describe()

In [ ]:
df.groupby("species").get_group("Gentoo").describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 4))

sns.scatterplot(x="bill_length_mm", y="flipper_length_mm", data=df, hue="species")
plt.title("Relacion largo de pico - largo de aleta")
plt.xlabel("Largo de pico")
plt.ylabel("Largo de aleta")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]], test_size=0.2, random_state=42, stratify=df["species"])

In [ ]:
train.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
esc = scaler.fit_transform(train)

pca = PCA(n_components=2)
components = pca.fit_transform(esc)

print(pca.explained_variance_ratio_)
print(pca.explained_variance_ratio_.sum())

loadings = pd.DataFrame(pca.components_, columns=train.columns, index=["PC1", "PC2"])
loadings

La primera combinacion de las cuatro medidas recoge el 69% de la variacion conjunta de los datos.

Mientras que la segunda combinacion recoge el 19% que sigue en importancia.

Juntas explican el 88% de como varian estas cuatro variables entre los pinguinos.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ks = range(2, 7)

inertias = []
scores = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42)
    etiquetes = km.fit_predict(components)
    inertias.append(km.inertia_)
    scores.append(silhouette_score(components, etiquetes))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].plot(ks, inertias, marker='o')
axes[0].set_title("Elbow")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")

axes[1].plot(ks, scores, marker='o')
axes[1].set_title("Silhouette")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette score")

plt.tight_layout()
plt.show()



La prueba del codo y de silueta no eligen el mismo k.

Partir en dos deja los pinguinos mas claramente de un lado o del otro. Agregar el tercer grupo afloja un poco esa nitidez, pero la caide de la inercia dice que ese tercer centro esta recogiendo una estructura que con dos grupos quedaba metida dentro de una misma nube. A partir de 4 la inercia casi no mejora y la silueta empeora.

K oscila entre 2(Silueta) o 3(Codo)

In [ ]:
k = 3

km = KMeans(n_clusters=k, random_state=42)
groups = km.fit_predict(components)

species = df.loc[train.index, "species"]

cross_tab = pd.crosstab(groups, species)
cross_tab